In [1]:
import os
from pathlib import Path
import shutil

TEXBIN = "/Library/TeX/texbin"
if Path(TEXBIN).is_dir():
    os.environ["PATH"] = TEXBIN + ":" + os.environ.get("PATH", "")

print("pdflatex now:", shutil.which("pdflatex"))
print("latexmk now:", shutil.which("latexmk"))

pdflatex now: /Library/TeX/texbin/pdflatex
latexmk now: /Library/TeX/texbin/latexmk


### User input & data generation

In [2]:
# ===========================
# VERSION 2.0 — WORKING BASELINE
# Extended abstract generator
# Topic-agnostic variable handling
# LaTeX-safe postprocessing
# Date: 02/05/2026
# ===========================

In [3]:
from openai import OpenAI
import numpy as np
import pandas as pd
import re, json, os, subprocess
from pathlib import Path
from dotenv import load_dotenv

# -----------------------------
# 0) Experiment configuration
# -----------------------------
load_dotenv()

experiment = {
    # This title is used exactly like your sample PDF (top of page, centered).
    # In the sample, the title wrapped into two lines. We preserve that style by allowing a manual line break.
    "doc_title": "Phosphate Uptake Kinetics and Equilibrium in Saline Soils: Batch\nTest Results",

    "context": (
        "Write an observations-style report of phosphate adsorption and uptake kinetics in saline soils "
        "using batch tests. Each run records an initial phosphate concentration, an equilibrium concentration, "
        "an equilibrium adsorption capacity, and time-series uptake values. Adsorption behavior is summarized "
        "using an equilibrium isotherm framework and a kinetic uptake framework, and adsorption favorability "
        "is indicated using a standard dimensionless factor. Describe what was done and what was recorded, "
        "not how data were generated."
    ),
    "motivation": (
        "Provide a clear lab-note summary: state the number of runs, the number of time points per run, recorded ranges "
        "and summary statistics, and describe how uptake changed with time across runs with different conditions."
    ),

    # RUNS (not total rows)
    "NUM_DATASETS": 1,
    "N_RUNS": 10,

    # time-series sampling points per run
    "times_min": [0, 2, 5, 10, 20, 30, 45, 60, 90, 120],

    # --- Controlled / set per run (ranges) ---
    "inputs": {
        "C0_(mg/L)"        : ("mg/L", "uniform", [5.0, 80.0]),          # initial phosphate concentration
        "m_(g)"            : ("g",    "uniform", [0.20, 1.00]),         # soil/adsorbent mass
        "V_(L)"            : ("L",    "uniform", [0.020, 0.100]),       # solution volume
        "qmax_(mg/g)"      : ("mg/g", "uniform", [0.50, 10.0]),         # Langmuir capacity
        "KL_(L/mg)"        : ("L/mg", "uniform", [0.01, 0.50]),         # Langmuir affinity
        "k2_(1/(mg·min))"  : ("1/(mg·min)", "uniform", [0.005, 0.20]),  # PSO rate constant
    },

    # --- Equations used internally (for your record) ---
    "equations": {
        "Langmuir_isotherm": "qe = (qmax * KL * Ce) / (1 + KL * Ce)",
        "Mass_balance"     : "C0 = Ce + (m/V) * qe",
        "Capacity_calc"    : "qe = ((C0 - Ce) * V) / m",
        "PSO_kinetics"     : "qt(t) = (k2 * qe^2 * t) / (1 + k2 * qe * t)",
        "Separation_factor": "RL = 1 / (1 + KL * C0)",
    },

    # --- Measurement-like noise model ---
    "noise": {
        "Ce_sd_frac_of_C0": 0.02,   # 2% of C0
        "qt_sd_frac_of_qe": 0.03,   # 3% of qe
    },

    # IMPORTANT: This controls ONLY the second table (full dataset at end).
    "INCLUDE_FULL_TABLE": False,

    "paper_words": 1000,
    "tok_paper": 2600,
    "model_id": "gpt-4.1",
    "out_dir": "Output/ExtendedAbstract_run",
}

# Updated to match your sample formatting:
# - No \ref usage. The sample uses "Table 1" and "Table 2" directly.
experiment["constraints"] = (
    "[INSTRUCTIONS—DO NOT INCLUDE THIS PARAGRAPH IN THE PAPER]\n"
    "Write in PASSIVE VOICE ONLY.\n"
    "Do NOT use any sectioning commands (no \\section, \\subsection, etc.).\n"
    "Do NOT use any LaTeX environments like abstract.\n"
    "The first paragraph must summarize the work WITHOUT using any variable names.\n"
    "From the second paragraph onward, variables may be used but must be italic and correctly subscripted.\n"
    "Do NOT include any equations or explicit functional forms.\n"
    "No bullet lists.\n"
    "Insert the token [[TABLE_HERE]] exactly once on its OWN LINE (no other text on that line).\n"
    "Do NOT write [[TABLE_HERE]] inside a sentence.\n"
    "If the table is mentioned, refer to it as 'Table 1' (and 'Table 2' if present).\n"
    f"Use the actual number of runs N={experiment['N_RUNS']}.\n"
    f"Use the actual number of time points per run T={len(experiment['times_min'])}.\n"
    f"Use the total number of recorded rows N_total={experiment['N_RUNS'] * len(experiment['times_min'])}.\n"
)

client = OpenAI()
rng = np.random.default_rng(2025)

OUT = Path(experiment["out_dir"])
OUT.mkdir(parents=True, exist_ok=True)

# -----------------------------
# 1) Utility: JSON robustness
# -----------------------------
def _extract_json_object(text: str) -> str | None:
    start = text.find("{")
    if start == -1:
        return None
    depth = 0
    for i in range(start, len(text)):
        ch = text[i]
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return text[start:i+1]
    return None

def _repair_json_loose(s: str) -> str:
    if not s:
        return s
    s = s.replace("“", '"').replace("”", '"').replace("’", "'").replace("‘", "'")
    s = re.sub(r",\s*([}\]])", r"\1", s)

    out, in_str, esc = [], False, False
    for ch in s:
        if in_str:
            if esc:
                out.append(ch); esc = False
            else:
                if ch == "\\":
                    out.append(ch); esc = True
                elif ch == '"':
                    out.append(ch); in_str = False
                elif ch == "\n":
                    out.append("\\n")
                else:
                    out.append(ch)
        else:
            out.append(ch)
            if ch == '"':
                in_str = True
    return "".join(out)

def _json_from_llm(text: str) -> dict:
    block = _extract_json_object(text)
    if not block:
        return {}
    try:
        return json.loads(block)
    except json.JSONDecodeError:
        fixed = _repair_json_loose(block)
        try:
            return json.loads(fixed)
        except json.JSONDecodeError:
            return {}

# -----------------------------
# 2) Data generation (physics-correct)
# -----------------------------
dists = {
    "uniform": lambda a, b, n: rng.uniform(a, b, n),
    "normal":  lambda m, s, n: rng.normal(m, s, n),
}

def sample_run_inputs(n_runs: int) -> pd.DataFrame:
    return pd.DataFrame({
        col: dists[dist](*params, n_runs)
        for col, (_, dist, params) in experiment["inputs"].items()
    })

def solve_Ce_from_langmuir_mass_balance(C0, qmax, KL, m, V):
    """
    Solve for Ce using:
        qe = (qmax * KL * Ce) / (1 + KL * Ce)
        C0 = Ce + (m/V) * qe
    Quadratic:
        a*Ce^2 + b*Ce + c = 0
        a = KL
        b = 1 + (m/V)*qmax*KL - C0*KL
        c = -C0
    """
    C0 = np.asarray(C0, dtype=float)
    qmax = np.asarray(qmax, dtype=float)
    KL = np.asarray(KL, dtype=float)
    m = np.asarray(m, dtype=float)
    V = np.asarray(V, dtype=float)

    a = KL
    b = 1.0 + (m / V) * qmax * KL - C0 * KL
    c = -C0

    disc = np.maximum(b*b - 4*a*c, 0.0)
    sqrt_disc = np.sqrt(disc)

    Ce1 = (-b + sqrt_disc) / (2*a)
    Ce2 = (-b - sqrt_disc) / (2*a)

    candidates = np.stack([Ce1, Ce2], axis=0)
    candidates = np.where(np.isfinite(candidates), candidates, np.nan)

    in_range = (candidates >= 0.0) & (candidates <= np.maximum(C0, 0.0))
    Ce = np.where(np.any(in_range, axis=0),
                  np.nanmax(np.where(in_range, candidates, np.nan), axis=0),
                  np.nan)

    nonneg = (candidates >= 0.0)
    Ce = np.where(np.isnan(Ce) & np.any(nonneg, axis=0),
                  np.nanmax(np.where(nonneg, candidates, np.nan), axis=0),
                  Ce)

    Ce = np.where(np.isnan(Ce), np.maximum(C0, 0.0), Ce)
    return Ce

def langmuir_qe(Ce, qmax, KL):
    Ce = np.asarray(Ce, dtype=float)
    return (qmax * KL * Ce) / (1.0 + KL * Ce)

def separation_factor_RL(C0, KL):
    C0 = np.asarray(C0, dtype=float)
    return 1.0 / (1.0 + KL * C0)

def pseudo_second_order_qt(t, qe, k2):
    t = np.asarray(t, dtype=float)
    qe = np.asarray(qe, dtype=float)
    k2 = np.asarray(k2, dtype=float)
    return (k2 * (qe**2) * t) / (1.0 + k2 * qe * t)

def generate_timeseries_dataset() -> pd.DataFrame:
    n_runs = experiment["N_RUNS"]
    times = np.array(experiment["times_min"], dtype=float)

    run_df = sample_run_inputs(n_runs)

    C0   = run_df["C0_(mg/L)"].to_numpy()
    m    = run_df["m_(g)"].to_numpy()
    V    = run_df["V_(L)"].to_numpy()
    qmax = run_df["qmax_(mg/g)"].to_numpy()
    KL   = run_df["KL_(L/mg)"].to_numpy()
    k2   = run_df["k2_(1/(mg·min))"].to_numpy()

    # TRUE equilibrium based on Langmuir + mass balance
    Ce_true = solve_Ce_from_langmuir_mass_balance(C0=C0, qmax=qmax, KL=KL, m=m, V=V)
    qe_true = langmuir_qe(Ce_true, qmax=qmax, KL=KL)

    # Measurement-like Ce noise, clipped to physical limits
    Ce_sd = experiment["noise"]["Ce_sd_frac_of_C0"] * np.maximum(C0, 1e-6)
    Ce_obs = Ce_true + rng.normal(0.0, Ce_sd, size=n_runs)
    Ce_obs = np.clip(Ce_obs, 0.0, C0)

    # Observed qe computed from capacity calc (like a lab calculation)
    qe_obs = ((C0 - Ce_obs) * V) / m
    qe_obs = np.clip(qe_obs, 0.0, None)

    RL = separation_factor_RL(C0, KL)

    rows = []
    for i in range(n_runs):
        qt_true = pseudo_second_order_qt(times, qe=qe_true[i], k2=k2[i])

        qt_sd = experiment["noise"]["qt_sd_frac_of_qe"] * max(qe_true[i], 1e-6)
        qt_obs = qt_true + rng.normal(0.0, qt_sd, size=len(times))

        # Physical cap: qt should not exceed qe_obs
        cap = max(qe_obs[i], 1e-9)
        qt_obs = np.clip(qt_obs, 0.0, cap)

        for tj, qtj in zip(times, qt_obs):
            rows.append({
                "run_id": i + 1,
                "t_(min)": float(tj),

                # observed (treated as recorded in a lab)
                "C0_(mg/L)": float(C0[i]),
                "Ce_(mg/L)": float(Ce_obs[i]),
                "qe_(mg/g)": float(qe_obs[i]),
                "qt_(mg/g)": float(qtj),

                # run descriptors (recorded per run)
                "qmax_(mg/g)": float(qmax[i]),
                "KL_(L/mg)"  : float(KL[i]),
                "k2_(1/(mg·min))": float(k2[i]),
                "RL_(-)"     : float(RL[i]),
                "m_(g)"      : float(m[i]),
                "V_(L)"      : float(V[i]),
            })

    return pd.DataFrame(rows).round(6)

def stats_json(df: pd.DataFrame) -> str:
    """
    Summary stats for numeric columns (excluding run_id).
    """
    df2 = df.copy()
    if "run_id" in df2.columns:
        df2 = df2.drop(columns=["run_id"])
    return json.dumps(
        df2.describe().T[["mean", "std", "min", "max"]].round(4).to_dict(),
        indent=2
    )

# -----------------------------
# 3) Variable math formatting in BODY (keep like your previous pipeline)
# -----------------------------
def parse_col_key(key: str):
    m = re.match(r"^(.+?)_\((.+)\)$", key)
    if m:
        return m.group(1), m.group(2)
    return key, ""

def symbol_to_math(symbol: str) -> str:
    s = symbol.strip()
    if "_" in s:
        base, sub = s.split("_", 1)
        sub = sub.strip("{}")
        return rf"$\mathit{{{base}}}_{{{sub}}}$"
    if re.fullmatch(r"[A-Za-z]", s):
        return rf"$\mathit{{{s}}}$"
    m = re.fullmatch(r"([A-Za-z]+)([0-9]+)", s)
    if m:
        return rf"$\mathit{{{m.group(1)}}}_{{{m.group(2)}}}$"
    m = re.fullmatch(r"([A-Za-z])([A-Za-z0-9]+)", s)
    if m:
        return rf"$\mathit{{{m.group(1)}}}_{{{m.group(2)}}}$"
    return rf"$\mathit{{{s}}}$"

def build_var_map_from_df_cols(cols: list[str]) -> dict:
    varmap = {}
    for k in cols:
        sym, _unit = parse_col_key(k)
        varmap[sym] = {"math": symbol_to_math(sym)}
    return varmap

def normalize_broken_variable_spellings(text: str) -> str:
    text = re.sub(r"\b([A-Za-z])\s+([A-Za-z0-9])\b", r"\1\2", text)
    return text

def canonicalize_emph_symbol(s: str) -> str:
    if not s:
        return s
    x = s.strip()
    x = re.sub(r"[.,;:]+$", "", x)
    x = re.sub(r"\s+", "", x)
    x = re.sub(r"\\+_", "_", x)
    m = re.fullmatch(r"([A-Za-z]+)_\{?([A-Za-z0-9]+)\}?", x)
    if m:
        return f"{m.group(1)}{m.group(2)}"
    return x

def convert_emph_variable_to_math(inner_raw: str) -> str | None:
    if not inner_raw:
        return None
    x = inner_raw.strip()
    if len(x) > 30 or re.search(r"\s", x):
        return None
    x = re.sub(r"[.,;:]+$", "", x)
    x = re.sub(r"\\+_", "_", x)
    x = x.replace("{", "").replace("}", "")

    if "_" in x:
        base, sub = x.split("_", 1)
        if base and sub:
            return rf"$\mathit{{{base}}}_{{{sub}}}$"

    m = re.fullmatch(r"([A-Za-z]+)([0-9]+)", x)
    if m:
        return rf"$\mathit{{{m.group(1)}}}_{{{m.group(2)}}}$"

    m = re.fullmatch(r"([A-Za-z])([A-Za-z0-9]+)", x)
    if m:
        return rf"$\mathit{{{m.group(1)}}}_{{{m.group(2)}}}$"

    if re.fullmatch(r"[A-Za-z]", x):
        return rf"$\mathit{{{x}}}$"
    return None

_MATH_INLINE_RE = re.compile(r"(\$.*?\$)", re.S)

def apply_variable_math(text: str, VARMAP: dict, VARS: list[str]) -> str:
    if not text:
        return text
    text = normalize_broken_variable_spellings(text)

    def repl_emph(m):
        inner_raw = m.group(1)
        math = convert_emph_variable_to_math(inner_raw)
        if math:
            canon = canonicalize_emph_symbol(inner_raw)
            if canon in VARMAP:
                return VARMAP[canon]["math"]
            return math
        return r"\emph{" + inner_raw + "}"

    text = re.sub(r"\\emph\{([^{}]*)\}", repl_emph, text)

    parts = _MATH_INLINE_RE.split(text)
    for i, chunk in enumerate(parts):
        if not chunk:
            continue
        if chunk.startswith("$") and chunk.endswith("$"):
            continue
        for sym in VARS:
            rep = VARMAP[sym]["math"]
            chunk = re.sub(rf"\b{re.escape(sym)}\b", lambda m, r=rep: r, chunk)
        chunk = re.sub(r"(?<!\\)_", r"\\_", chunk)
        parts[i] = chunk
    return "".join(parts)

def remove_variables_from_first_paragraph(body: str, VARS: list[str]) -> str:
    parts = re.split(r"\n\s*\n", body, maxsplit=1)
    first = parts[0]
    rest  = parts[1] if len(parts) > 1 else ""
    for sym in VARS:
        first = re.sub(rf"\b{re.escape(sym)}\b", "the measured quantity", first)
        first = re.sub(rf"\\emph\{{\s*{re.escape(sym)}\s*\}}", "the measured quantity", first)
    return first + ("\n\n" + rest if rest else "")

# -----------------------------
# 4) LLM calls (prompt adjusted to match sample)
# -----------------------------
def llm_extended_body(stats: str) -> str:
    prompt = (
        f"{experiment['context']}\n\n"
        f"Motivation: {experiment['motivation']}\n\n"
        f"{experiment['constraints']}\n\n"
        "TABLE RULES (STRICT):\n"
        "- Insert [[TABLE_HERE]] exactly once on its own line.\n"
        "- If the table is mentioned, use the exact plain text 'Table 1' (and 'Table 2' if present).\n"
        "- Do NOT include equations.\n\n"
        f"Dataset summary statistics (JSON):\n{stats}\n\n"
        f"Write an ~{experiment['paper_words']}-word LaTeX write-up as connected paragraphs.\n"
        "Do NOT include section headings.\n"
        "Do NOT include any table environments; the table will be injected later.\n"
        "Use \\emph{} for variables when they are used (but only from paragraph 2 onward).\n"
    )
    res = client.chat.completions.create(
        model=experiment["model_id"],
        messages=[{"role": "user", "content": prompt}],
        max_tokens=experiment["tok_paper"],
        temperature=0.4,
    )
    return res.choices[0].message.content.strip()

def llm_table_caption(table_no: int, context: str) -> str:
    """
    Produces a short one-sentence caption like your sample:
    'Table 1: ...'
    """
    prompt = (
        "Return JSON ONLY with key 'caption'.\n"
        "Write a SHORT caption (12–20 words). ONE sentence only, passive voice.\n"
        "Do NOT include variable lists or ranges.\n"
        "Do NOT say generated/simulated/synthetic.\n\n"
        f"Table number: {table_no}\n"
        f"Context:\n{context}\n"
    )
    res = client.chat.completions.create(
        model=experiment["model_id"],
        messages=[{"role":"user","content":prompt}],
        max_tokens=120,
        temperature=0.2,
    )
    data = _json_from_llm(res.choices[0].message.content)
    cap = (data.get("caption") or "").strip()
    if not cap:
        cap = "Phosphate uptake was monitored over time in saline soils, recording concentration and uptake."

    cap = re.sub(r"\s+", " ", cap).strip()
    if cap.endswith("."):
        cap = cap[:-1].strip()

    return f"Table {table_no}: {cap}."

# -----------------------------
# 5) Tables + injection (match sample style)
# -----------------------------
def pretty_plain_header_from_key(key: str) -> str:
    """
    Match the sample table header style like:
    t ( min )   qe ( mg/g )   ...
    """
    sym, unit = parse_col_key(key)
    unit = (unit or "").strip()
    # put spaces similar to the sample: "t ( min )"
    if unit:
        return f"{sym} ( {unit} )"
    return sym

def df_to_sample_style_summary_table(df: pd.DataFrame, caption_line: str, table_no: int) -> str:
    """
    Produces a simple, non-floating table:
    Table 1: caption...
    <header row>
    Mean ...
    SD ...
    Min ...
    Max ...
    """
    df2 = df.copy()
    if "run_id" in df2.columns:
        df2 = df2.drop(columns=["run_id"])

    desc = df2.describe().loc[["mean", "std", "min", "max"]].copy()
    desc.index = ["Mean", "SD", "Min", "Max"]

    cols = list(desc.columns)
    header_cells = [pretty_plain_header_from_key(c) for c in cols]
    header = " & ".join(header_cells)

    lines = []
    lines.append(rf"\noindent {caption_line}")
    lines.append(r"\vspace{2pt}")
    lines.append(r"\begin{center}")
    lines.append(r"\small")
    lines.append(r"\begin{tabular}{l" + "c"*len(cols) + r"}")
    lines.append(header + r" \\")
    for idx, row in desc.iterrows():
        # match the sample: 2 decimals
        vals = " & ".join(f"{float(v):.2f}" for v in row.values)
        lines.append(f"{idx} & {vals} \\\\")
    lines.append(r"\end{tabular}")
    lines.append(r"\end{center}")
    lines.append(r"\vspace{2pt}")
    return "\n".join(lines)

def df_to_sample_style_full_table(df: pd.DataFrame, caption_line: str, table_no: int, max_rows: int | None = None) -> str:
    """
    Full table like sample Table 2:
    Table 2: ...
    Run t (...) C0 (...) ...
    rows...
    """
    df2 = df.copy()
    if max_rows is not None:
        df2 = df2.head(max_rows)

    cols = list(df2.columns)
    header_cells = []
    for c in cols:
        if c == "run_id":
            header_cells.append("Run")
        else:
            header_cells.append(pretty_plain_header_from_key(c))
    header = " & ".join(header_cells)

    lines = []
    lines.append(rf"\noindent {caption_line}")
    lines.append(r"\vspace{2pt}")
    lines.append(r"\begin{center}")
    lines.append(r"\tiny")
    lines.append(r"\begin{tabular}{" + "c"*len(cols) + r"}")
    lines.append(header + r" \\")
    for _, row in df2.iterrows():
        vals = []
        for c in cols:
            if c == "run_id":
                vals.append(str(int(row[c])))
            else:
                vals.append(f"{float(row[c]):.4f}")
        lines.append(" & ".join(vals) + r" \\")
    lines.append(r"\end{tabular}")
    lines.append(r"\end{center}")
    lines.append(r"\vspace{2pt}")
    return "\n".join(lines)

def ensure_marker_own_line(body: str) -> str:
    body = re.sub(r"[ \t]*\[\[TABLE_HERE\]\][ \t]*", "[[TABLE_HERE]]", body)
    body = re.sub(r"(?!\n)\[\[TABLE_HERE\]\](?!\n)", "\n\n[[TABLE_HERE]]\n\n", body)
    return body

def inject_table(body: str, table_tex: str) -> str:
    body = ensure_marker_own_line(body)
    if "[[TABLE_HERE]]" in body:
        return body.replace("[[TABLE_HERE]]", table_tex, 1)
    parts = re.split(r"\n\s*\n", body, maxsplit=1)
    if len(parts) == 2:
        return parts[0] + "\n\n" + table_tex + "\n\n" + parts[1]
    return body + "\n\n" + table_tex

def strip_table_marker(body: str) -> str:
    return body.replace("[[TABLE_HERE]]", "").replace("[[TABLE\\_HERE]]", "")

# -----------------------------
# 6) LaTeX document builder + compile (match sample layout)
# -----------------------------
def build_tex_doc(title_text: str, body: str) -> str:
    # Match sample: centered title (can be 2 lines), then "Extended Abstract"
    # No \maketitle.
    safe_title = title_text.replace("&", r"\&")
    return r"""\documentclass[10pt]{article}
\usepackage[margin=1in]{geometry}
\usepackage{microtype}
\setlength{\parskip}{6pt}
\setlength{\parindent}{0pt}

\begin{document}

\begin{center}
""" + r"\textbf{" + safe_title + r"}" + r"""
\end{center}

\begin{center}
\textbf{Extended Abstract}
\end{center}

""" + body + r"""

\end{document}
"""

def ensure_tex_on_path():
    candidates = ["/Library/TeX/texbin", "/usr/texbin", "/usr/local/bin", "/usr/bin"]
    path_parts = os.environ.get("PATH", "").split(os.pathsep)
    for p in candidates:
        if p and os.path.isdir(p) and p not in path_parts:
            path_parts.insert(0, p)
    os.environ["PATH"] = os.pathsep.join(path_parts)

def compile_tex(tex_path: Path, workdir: Path):
    ensure_tex_on_path()
    proc = subprocess.run(
        ["pdflatex", "-interaction=nonstopmode", "-halt-on-error", tex_path.name],
        cwd=workdir,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    return proc.returncode == 0, proc.stdout

def show_first_latex_error(log_text: str, max_lines: int = 60) -> str:
    lines = (log_text or "").splitlines()
    for i, ln in enumerate(lines):
        if ln.startswith("!"):
            start = max(0, i - 5)
            end = min(len(lines), i + max_lines)
            return "\n".join(lines[start:end])
    return "\n".join(lines[:max_lines])

# -----------------------------
# 7) Main loop
# -----------------------------
for i in range(1, experiment["NUM_DATASETS"] + 1):
    print(f"▶ Dataset {i}/{experiment['NUM_DATASETS']}")

    # Generate physics-correct batch time-series dataset
    df = generate_timeseries_dataset()
    stats = stats_json(df)

    # Variable map from actual dataframe columns (excluding run_id)
    cols_for_varmap = [c for c in df.columns if c != "run_id"]
    VARMAP = build_var_map_from_df_cols(cols_for_varmap)
    VARS = sorted(VARMAP.keys(), key=len, reverse=True)

    # LLM write-up (table placeholder inserted once)
    body_raw = llm_extended_body(stats)

    # Remove variable names in first paragraph
    body_raw = remove_variables_from_first_paragraph(body_raw, VARS)

    # Convert variable mentions to math italics in the BODY only
    body = apply_variable_math(body_raw, VARMAP, VARS)

    # Build + inject Table 1 (summary stats)
    cap1 = llm_table_caption(1, experiment["context"])
    table1_tex = df_to_sample_style_summary_table(df, cap1, table_no=1)
    body = inject_table(body, table1_tex)

    # Optionally append Table 2 (full table at end)
    if experiment.get("INCLUDE_FULL_TABLE", False):
        cap2 = llm_table_caption(2, experiment["context"])
        # Full table can be large; keep as-is or cap rows if you want.
        table2_tex = df_to_sample_style_full_table(df, cap2, table_no=2, max_rows=None)
        body = strip_table_marker(body)
        body = body + "\n\n" + table2_tex
    else:
        body = strip_table_marker(body)

    # Build TeX with fixed title like the sample
    tex_doc = build_tex_doc(experiment["doc_title"], body)
    assert tex_doc.count("{") == tex_doc.count("}"), "Brace mismatch before pdflatex"

    tex_out = OUT / f"paper{i:02d}.tex"
    tex_out.write_text(tex_doc, encoding="utf-8")

    ok, log = compile_tex(tex_out, OUT)
    if not ok:
        (OUT / f"paper{i:02d}.log.txt").write_text(log, encoding="utf-8")
        print(f"   ✖ compile failed; saved log to paper{i:02d}.log.txt")
        print("\n--- First LaTeX error snippet ---")
        print(show_first_latex_error(log))
        raise RuntimeError("pdflatex failed; inspect the snippet above and the saved log.")

    pdf_path = OUT / f"paper{i:02d}.pdf"
    if not pdf_path.exists():
        raise RuntimeError("Compilation succeeded but PDF not found where expected.")

    print("   ✔ saved", pdf_path)
    print("   ✔ saved", tex_out)

print("🎉 Done. Files saved in:", OUT.resolve())

▶ Dataset 1/1
   ✔ saved Output/ExtendedAbstract_run/paper01.pdf
   ✔ saved Output/ExtendedAbstract_run/paper01.tex
🎉 Done. Files saved in: /Users/md_halim_mondol/Desktop/STEPS-LLM/steps-llm/Src/Output/ExtendedAbstract_run
